<a href="https://colab.research.google.com/github/shahriarshadhin3-alt/uterine-emg-preterm-detection/blob/main/notebooks/tpehg_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wfdb PyWavelets scipy scikit-learn xgboost imbalanced-learn matplotlib seaborn

In [ ]:
from google.colab import files
import os

# 1. Upload your kaggle.json
print("Upload your kaggle.json:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Download the PhysioNet Term-Preterm EHG Database v1.0.1
!kaggle datasets download -d zubayer09/zubayer-term-preterm-ehg-database-1-0-1 -p ./dataset --unzip

# Verify contents
!ls -lh ./dataset

Upload your kaggle.json:


In [ ]:
import glob
import re
import wfdb
import numpy as np
import pandas as pd

# Locate directory containing .hea and .dat files
hea_files = glob.glob('./dataset/**/*.hea', recursive=True)
record_paths = [os.path.splitext(f)[0] for f in hea_files if not f.endswith('tpehgdb.hea')]

print(f"Total EHG records found: {len(record_paths)}")

def parse_header_metadata(record_path):
    """Extract clinical target and recording time from .hea comment lines."""
    _, meta = wfdb.rdsamp(record_path)
    comments = " ".join(meta.get('comments', []))

    # Check premature status
    premature_match = re.search(r'Premature\s*:\s*([tf])', comments, re.IGNORECASE)
    if premature_match:
        label = 1 if premature_match.group(1).lower() == 't' else 0
    else:
        # Fallback to delivery duration < 37 weeks
        gest_match = re.search(r'Gestation\s*:\s*([\d\.]+)', comments, re.IGNORECASE)
        label = 1 if gest_match and float(gest_match.group(1)) < 37.0 else 0

    early_match = re.search(r'Early\s*:\s*([tf])', comments, re.IGNORECASE)
    is_early = 1 if early_match and early_match.group(1).lower() == 't' else 0

    return label, is_early, meta['fs']

In [ ]:
from scipy.signal import butter, filtfilt, welch
from scipy.stats import skew, kurtosis
import pywt

def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return b, a

def filter_ehg(signal_data, lowcut=0.34, highcut=3.0, fs=20.0):
    b, a = butter_bandpass(lowcut, highcut, fs)
    return filtfilt(b, a, signal_data, axis=0)

def extract_features(segment, fs=20.0):
    feats = {}

    # 1. Wavelet Decomposition (DWT: 4-level db4)
    coeffs = pywt.wavedec(segment, 'db4', level=4)
    for idx, c in enumerate(coeffs):
        feats[f'wavelet_energy_L{idx}'] = np.sum(c ** 2) / len(c)
        feats[f'wavelet_std_L{idx}'] = np.std(c)

    # 2. Continuous Wavelet Transform (CWT Energy using Morlet)
    scales = np.arange(1, 32)
    cwtmatr, _ = pywt.cwt(segment, scales, 'morl')
    feats['cwt_mean_energy'] = np.mean(cwtmatr ** 2)
    feats['cwt_max_energy'] = np.max(cwtmatr ** 2)

    # 3. Frequency & Spectral Features
    freqs, psd = welch(segment, fs=fs, nperseg=min(len(segment), 256))
    norm_psd = psd / (np.sum(psd) + 1e-12)
    feats['f_peak'] = freqs[np.argmax(psd)]
    feats['f_med'] = freqs[np.where(np.cumsum(norm_psd) >= 0.5)[0][0]]
    feats['spectral_entropy'] = -np.sum(norm_psd * np.log2(norm_psd + 1e-12))

    # 4. Amplitude Dynamics
    feats['rms'] = np.sqrt(np.mean(segment ** 2))
    feats['skewness'] = skew(segment)
    feats['kurtosis'] = kurtosis(segment)

    return feats

In [ ]:
rows = []
window_seconds = 120

for path in record_paths:
    try:
        sig, meta = wfdb.rdsamp(path)
        label, is_early, fs = parse_header_metadata(path)
        rec_id = os.path.basename(path)

        # Channel 3 (S3 = E4 - E3) is the standard abdominal reference
        ch3 = sig[:, 2]
        filtered_ch3 = filter_ehg(ch3, lowcut=0.34, highcut=3.0, fs=fs)

        win_len = int(window_seconds * fs)
        for w_idx, start in enumerate(range(0, len(filtered_ch3) - win_len, win_len)):
            seg = filtered_ch3[start:start + win_len]
            feat_dict = extract_features(seg, fs=fs)
            feat_dict['record_id'] = rec_id
            feat_dict['is_early'] = is_early
            feat_dict['target'] = label
            rows.append(feat_dict)
    except Exception as e:
        continue

df_dataset = pd.DataFrame(rows)
print(f"Extraction complete! Feature Matrix Shape: {df_dataset.shape}")
print(df_dataset['target'].value_counts())

# Save locally to repository
df_dataset.to_csv("tpehg_time_frequency_features.csv", index=False)

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

X = df_dataset.drop(columns=['target', 'record_id', 'is_early'])
y = df_dataset['target']
groups = df_dataset['record_id']

gkf = GroupKFold(n_splits=5)
rf_aucs, xgb_aucs = [], []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Scale within fold
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Handle class imbalance (Term vs Preterm)
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

    # Model 1: Random Forest
    rf = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
    rf.fit(X_train_res, y_train_res)
    rf_probs = rf.predict_proba(X_test_scaled)[:, 1]
    rf_aucs.append(roc_auc_score(y_test, rf_probs))

    # Model 2: XGBoost
    xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, eval_metric='logloss')
    xgb.fit(X_train_res, y_train_res)
    xgb_probs = xgb.predict_proba(X_test_scaled)[:, 1]
    xgb_aucs.append(roc_auc_score(y_test, xgb_probs))

print(f"5-Fold Patient-Grouped RF AUC-ROC:  {np.mean(rf_aucs):.4f}")
print(f"5-Fold Patient-Grouped XGB AUC-ROC: {np.mean(xgb_aucs):.4f}")

In [ ]:
!git config --global user.name "Your GitHub Username"
!git config --global user.email "your_email@example.com"

# Prevent pushing large raw dataset files
with open(".gitignore", "a") as f:
    f.write("\ndataset/\n*.hea\n*.dat\nkaggle.json\n")

!git add .gitignore tpehg_time_frequency_features.csv
!git commit -m "Implement TPEHG time-frequency feature extraction and GroupKFold evaluation"
!git push origin main

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

tprs = []
base_fpr = np.linspace(0, 1, 101)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train_s, y_train)

    model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, eval_metric='logloss')
    model.fit(X_res, y_res)

    y_score = model.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, lw=1, alpha=0.3, label=f'Fold {fold+1} (AUC = {roc_auc:.2f})')

    tpr_interp = np.interp(base_fpr, fpr, tpr)
    tpr_interp[0] = 0.0
    tprs.append(tpr_interp)

mean_tprs = np.mean(tprs, axis=0)
mean_tprs[-1] = 1.0
mean_auc = auc(base_fpr, mean_tprs)

plt.plot(base_fpr, mean_tprs, color='b', lw=2, label=f'Mean ROC (AUC = {mean_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('TPEHG Preterm Detection ROC Curve (GroupKFold)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.savefig('results_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!pip install shap -q
import shap

# Train model on full dataset for global feature attribution
scaler = StandardScaler()
X_s = scaler.fit_transform(X)
model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, eval_metric='logloss')
model.fit(X_s, y)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_s)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.title("TPEHG Feature Importance via SHAP")
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sub-cohort evaluation: Early recordings only
early_mask = (df_dataset['is_early'] == 1)
X_early = X[early_mask]
y_early = y[early_mask]
groups_early = groups[early_mask]

early_gkf = GroupKFold(n_splits=5)
early_aucs = []

for train_idx, test_idx in early_gkf.split(X_early, y_early, groups=groups_early):
    X_tr, X_te = X_early.iloc[train_idx], X_early.iloc[test_idx]
    y_tr, y_te = y_early.iloc[train_idx], y_early.iloc[test_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_tr_s, y_tr)

    clf = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, eval_metric='logloss')
    clf.fit(X_res, y_res)

    probs = clf.predict_proba(X_te_s)[:, 1]
    early_aucs.append(roc_auc_score(y_te, probs))

print(f"Early Recording Subgroup AUC-ROC: {np.mean(early_aucs):.4f}")

In [ ]:
# Push figures and updated files to GitHub
!git add results_roc_curve.png shap_summary.png
!git commit -m "Add cross-validation ROC curves and SHAP explainability plots"
!git push origin main